<div align="center">

# WEEK 03
## A6 DATA DICTIONARY & DOCUMENTATION

### CadetX Virtual Work Experience
### Heavy Supplier, Inventory & Warehouse Analytics

</div>

---

| | |
|---|---|
| **Prepared by** | Keren Oyem (Data Analyst / Data Scientist) |
| **Sprint** | Week 3 · Fri 6 Nov – Thu 12 Nov 2026 |
| **Catalogue task** | A6 Data Dictionary & Documentation |
| **Input** | `warehouse.duckdb` (schemas `main`, `star`, `feat`) and `data/processed/*.parquet` |
| **Output** | `docs/data_dictionary.xlsx` and `docs/data_dictionary.docx` |
| **Tools** | Python, SQL on DuckDB, pandas, openpyxl, python-docx |

## Purpose

A data dictionary tells anyone who opens the project **what every field means, where it comes from, what values it holds and what can go wrong with it**. The plan asks for a dictionary of *all raw and derived fields* (source, type, definition, allowed values, quality notes).

## What it covers

| Layer | Content | Source of the definition |
|---|---|---|
| Raw | 12 original tables | Hand-written definitions (this notebook) |
| Clean | Columns added in Week 2 (flags, numeric capacity, volume, recalculated values) | Hand-written definitions |
| Star | 5 dimensions and 4 facts | Descriptions stored in the database (Week 3) |
| Feature | 8 feature tables | Descriptions stored in the database (Week 3) |

## How it is built (and why it is trustworthy)

1. **Structure and descriptions are read from the database**, not typed again, so they cannot drift from the real tables.
2. **Values, ranges and null counts are measured live** for every column.
3. **Definitions** come from the database where they were stored in Weeks 2-3, otherwise from the definition library in section 1.
4. **Quality notes** come from the Week 2 issue log.
5. A **coverage check** confirms that no column is left without a definition.

## Honest limitation

`Fields Documentation.docx` was not available (see the plan, section 2.3). Raw master-table definitions are therefore based on column names and your Week 1-2 findings, and are marked **Inferred** until reconciled with the official guide. Fields checked in Weeks 1-2 (orders, invoices, payments, stock ledger, keys) are marked **Verified**.

## Contents

0. Setup · 1. Definition library · 2. Read the structure · 3. Measure values · 4. Combine definitions · 5. Tables overview · 6. Write the files · 7. Check · 8. Summary

## 0. Setup

**Objective.** Open the database read-only (nothing is changed) and confirm the star and feature schemas exist.

Run Jupyter from the **project folder**. If you see a *file locked* error, another notebook still has the database open for writing: restart that kernel.

In [1]:
import datetime
import importlib
import os
import sys

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

try:
    con.close()
except Exception:
    pass
con = duckdb.connect("warehouse.duckdb", read_only=True)

n_star = con.execute("SELECT COUNT(*) FROM duckdb_tables() WHERE schema_name = 'star'").fetchone()[0]
n_feat = con.execute("SELECT COUNT(*) FROM duckdb_tables() WHERE schema_name = 'feat'").fetchone()[0]
assert n_star == 9 and n_feat == 8, "Run 03_star_schema.ipynb and 03_feature_engineering.ipynb first."
os.makedirs("docs", exist_ok=True)
os.makedirs("src", exist_ok=True)
print(f"Connected (read-only). star tables: {n_star}, feat tables: {n_feat}")

Connected (read-only). star tables: 9, feat tables: 8


**Reading the output.** `Connected (read-only). star tables: 9, feat tables: 8`.

## 1. Definition library

**Objective.** Hold the hand-written knowledge that cannot be read from the database.

| Dictionary | What it holds |
|---|---|
| `RAW_DEFS` | One definition per raw column (`table.column`) |
| `CLEAN_DEFS` | Definitions of columns added by the Week 2 cleaning |
| `RAW_GRAIN` | What one row of each raw table represents |
| `QUALITY` | Known data-quality notes from the Week 2 issue log |
| `STAR_SOURCE` | Which raw table each dimension is copied from, so dimension columns inherit their definition |

**You can edit this cell.** When the official Fields Documentation arrives, correct any wrong definition here and re-run the notebook; both files are rebuilt.

In [2]:
# ---------------------------------------------------------------------------
# DEFINITION LIBRARY (hand-written). Keys are "table.column".
# Raw definitions are based on column names, the Week 1-2 profiling and the plan.
# Fields Documentation.docx was not available, so raw definitions of master
# tables are marked "Inferred" until reconciled with the official guide.
# ---------------------------------------------------------------------------
RAW_DEFS = {
 # branches
 "branches.branch_id": "Unique branch / warehouse code (e.g. DEL001).",
 "branches.branch_name": "Branch name.",
 "branches.city": "City where the branch is located.",
 "branches.state": "State where the branch is located.",
 "branches.region": "Sales region of the branch (North, South, East, West).",
 "branches.warehouse_type": "Type of warehouse (category label).",
 "branches.warehouse_capacity": "Warehouse capacity as text with its unit, e.g. '45230 sqft'.",
 "branches.service_center_available": "Whether the branch has a service centre (TRUE/FALSE).",
 "branches.manager_id": "Identifier of the branch manager.",
 "branches.total_employees": "Number of employees at the branch.",
 "branches.avg_monthly_revenue": "Average monthly revenue of the branch as stored in the branch master, INR.",
 "branches.monthly_operational_cost": "Monthly operating cost of the branch, INR.",
 "branches.market_demand_index": "Index of market demand around the branch (scale not documented).",
 # products
 "products.product_id": "Unique product (spare part) code.",
 "products.product_name": "Product name.",
 "products.category": "Product category.",
 "products.machine_type": "Type of heavy machine the part is used on.",
 "products.brand": "Brand of the part.",
 "products.model_compatibility": "Machine models the part is compatible with.",
 "products.unit_cost": "Cost per unit, INR before GST.",
 "products.unit_price": "Selling price per unit, INR before GST.",
 "products.margin_percentage": "Margin percentage as stored in the product master (definition not documented; compare with margin_pct_calc).",
 "products.gst_rate": "GST rate in percent (18 or 28).",
 "products.weight_kg": "Unit weight in kilograms.",
 "products.dimensions_cm": "Unit dimensions as text 'length x width x height' in centimetres.",
 "products.material_type": "Main material of the part.",
 "products.warranty_months": "Warranty period in months.",
 "products.reorder_level": "Company-level reorder level in units (branch-level values are in inventory_master).",
 "products.safety_stock": "Company-level safety stock in units.",
 "products.max_stock_level": "Company-level maximum stock level in units.",
 "products.lead_time_days": "Replenishment lead time in days as stored in the product master.",
 "products.criticality_level": "How critical the part is to machine operation (label).",
 "products.usage_frequency": "How often the part is used (label).",
 "products.uom": "Unit of measure.",
 "products.last_purchase_price": "Price paid in the most recent purchase, INR.",
 "products.last_purchase_date": "Date of the most recent purchase.",
 # inventory_master
 "inventory_master.product_id": "Product code (with branch_id forms the key).",
 "inventory_master.branch_id": "Branch code (with product_id forms the key).",
 "inventory_master.opening_stock": "Units in stock at the start of the ledger period.",
 "inventory_master.reorder_level": "Branch-level reorder level in units.",
 "inventory_master.safety_stock": "Branch-level safety stock in units.",
 "inventory_master.max_stock": "Branch-level maximum stock in units.",
 "inventory_master.current_stock": "Units in stock as recorded in the inventory master.",
 "inventory_master.warehouse_bin": "Bin location code in the warehouse (zones A-F).",
 # suppliers
 "suppliers.supplier_id": "Unique supplier code.",
 "suppliers.supplier_name": "Supplier name.",
 "suppliers.supplier_type": "Type of supplier (label).",
 "suppliers.product_category": "Product category the supplier provides.",
 "suppliers.city": "Supplier city.",
 "suppliers.province": "Supplier province in China.",
 "suppliers.region": "Supplier region.",
 "suppliers.pincode": "Supplier postal code.",
 "suppliers.lead_time_days": "Stated supplier lead time in days.",
 "suppliers.reliability_score": "Stated reliability score of the supplier (scale not documented).",
 "suppliers.import_duty_rate": "Import duty rate on goods from the supplier (assumed percent).",
 "suppliers.china_tax_id": "Supplier tax identifier in China.",
 # customers
 "customers.customer_id": "Unique customer code.",
 "customers.customer_type": "Type of customer (label).",
 "customers.industry_segment": "Industry the customer works in.",
 "customers.city": "Customer city.",
 "customers.state": "Customer state.",
 "customers.pincode": "Customer postal code.",
 "customers.region": "Customer region (North, South, East, West, Central, North East).",
 "customers.branch_id": "Branch that serves the customer.",
 "customers.credit_limit": "Credit limit granted to the customer, INR.",
 "customers.current_balance": "Current outstanding balance of the customer, INR.",
 "customers.payment_terms": "Payment terms agreed with the customer (label).",
 "customers.customer_since": "Date the customer relationship started.",
 "customers.last_purchase_date": "Date of the customer's last purchase as stored.",
 "customers.total_purchase_value": "Cumulative purchase value as stored, INR (unreliable; see notes).",
 "customers.customer_rating": "Customer rating (scale not documented).",
 # invoices
 "invoices.invoice_id": "Invoice code (not unique; see notes).",
 "invoices.so_id": "Sales order the invoice belongs to (unique per invoice).",
 "invoices.customer_id": "Customer invoiced.",
 "invoices.branch_id": "Branch that issued the invoice.",
 "invoices.invoice_date": "Date the invoice was issued.",
 "invoices.due_date": "Date payment is due.",
 "invoices.total_order_value": "Invoice value before GST, INR.",
 "invoices.total_gst_amount": "GST on the invoice, INR.",
 "invoices.grand_total": "Invoice value including GST, INR.",
 "invoices.payment_status": "Payment status as stored: Paid, Partially Paid or Unpaid (see notes).",
 # payments
 "payments.payment_id": "Payment code (not unique; see notes).",
 "payments.invoice_id": "Invoice the payment is applied to.",
 "payments.payment_date": "Date the payment was received.",
 "payments.payment_amount": "Amount received, INR.",
 "payments.payment_method": "How the payment was made (label).",
 # purchase_orders_header
 "purchase_orders_header.po_id": "Unique purchase order code.",
 "purchase_orders_header.supplier_id": "Supplier the order was placed with.",
 "purchase_orders_header.branch_id": "Branch receiving the order.",
 "purchase_orders_header.order_date": "Date the PO was placed.",
 "purchase_orders_header.expected_delivery_date": "Promised delivery date.",
 "purchase_orders_header.received_date": "Actual receipt date (empty for cancelled POs).",
 "purchase_orders_header.po_status": "PO status: Received or Cancelled.",
 "purchase_orders_header.total_cost": "PO value before GST, INR.",
 "purchase_orders_header.total_gst_amount": "GST on the PO, INR.",
 "purchase_orders_header.grand_total": "PO value including GST, INR.",
 # purchase_orders_lines
 "purchase_orders_lines.po_id": "Purchase order the line belongs to.",
 "purchase_orders_lines.line_number": "Line number within the PO.",
 "purchase_orders_lines.product_id": "Product ordered.",
 "purchase_orders_lines.quantity": "Units ordered.",
 "purchase_orders_lines.unit_cost": "Cost per unit on the PO, INR before GST.",
 "purchase_orders_lines.gst_rate": "GST rate in percent.",
 "purchase_orders_lines.line_total": "quantity x unit_cost, INR before GST.",
 "purchase_orders_lines.gst_amount": "GST on the line, INR.",
 "purchase_orders_lines.line_grand_total": "line_total + gst_amount, INR.",
 # sales_orders_header
 "sales_orders_header.so_id": "Unique sales order code.",
 "sales_orders_header.customer_id": "Customer who placed the order.",
 "sales_orders_header.branch_id": "Branch that fulfils the order.",
 "sales_orders_header.order_date": "Date the order was placed.",
 "sales_orders_header.delivery_date": "Delivery date (a placeholder for cancelled orders).",
 "sales_orders_header.order_status": "Order status: Delivered or Cancelled.",
 "sales_orders_header.payment_terms": "Payment terms on the order (label).",
 "sales_orders_header.total_order_value": "Order value before GST, INR.",
 "sales_orders_header.total_gst_amount": "GST on the order, INR.",
 "sales_orders_header.grand_total": "Order value including GST, INR.",
 "sales_orders_header.sales_channel": "Channel through which the order was placed (label).",
 # sales_orders_lines
 "sales_orders_lines.so_id": "Sales order the line belongs to.",
 "sales_orders_lines.line_number": "Line number within the order.",
 "sales_orders_lines.product_id": "Product sold.",
 "sales_orders_lines.quantity": "Units sold.",
 "sales_orders_lines.unit_price": "Selling price per unit, INR before GST (matches the product master).",
 "sales_orders_lines.gst_rate": "GST rate in percent (18 or 28).",
 "sales_orders_lines.line_total": "quantity x unit_price, INR before GST.",
 "sales_orders_lines.gst_amount": "GST on the line, INR.",
 "sales_orders_lines.line_grand_total": "line_total + gst_amount, INR.",
 # stock_ledger
 "stock_ledger.movement_id": "Unique stock movement code.",
 "stock_ledger.product_id": "Product moved.",
 "stock_ledger.branch_id": "Branch where the movement happened.",
 "stock_ledger.movement_type": "IN (receipt), OUT (issue) or ADJUSTMENT.",
 "stock_ledger.movement_date": "Date of the movement.",
 "stock_ledger.quantity": "Units moved; always positive (direction comes from movement_type).",
 "stock_ledger.reference_type": "Type of source document (PO, SO or ADJ).",
 "stock_ledger.reference_id": "ID of the source document.",
 "stock_ledger.running_balance": "Stock balance recorded after the movement.",
}

# Columns added or changed by the Week 2 cleaning layer (data/processed/*.parquet)
CLEAN_DEFS = {
 "branches.warehouse_capacity_sqft": "Numeric capacity in square feet, parsed from warehouse_capacity.",
 "products.length_cm": "Length in cm, parsed from dimensions_cm.",
 "products.width_cm": "Width in cm, parsed from dimensions_cm.",
 "products.height_cm": "Height in cm, parsed from dimensions_cm.",
 "products.volume_m3": "Unit volume in cubic metres (length x width x height / 1,000,000).",
 "inventory_master.current_above_max_flag": "TRUE when current_stock exceeds max_stock.",
 "customers.recalc_purchase_value_with_gst": "Purchase value recomputed from delivered sales orders, incl. GST, INR.",
 "customers.recalc_purchase_value_before_gst": "Purchase value recomputed from delivered sales orders, before GST, INR.",
 "customers.purchase_value_mismatch_flag": "TRUE when the stored purchase value differs from the recomputed value.",
 "invoices.total_paid": "Sum of payments received against the invoice_id, INR.",
 "invoices.payment_count": "Number of payments received against the invoice_id.",
 "invoices.outstanding_balance": "grand_total - total_paid, INR. Negative means overpaid.",
 "invoices.invoice_id_duplicated": "TRUE when the invoice_id is shared with another invoice.",
 "invoices.payment_status_derived": "Status derived from payments: Unpaid, Partially paid, Fully paid, Overpaid or Ambiguous.",
 "payments.payment_id_duplicated": "TRUE when the payment_id is shared with another payment.",
 "sales_orders_header.is_cancelled": "TRUE when the order status is Cancelled.",
 "sales_orders_header.days_to_deliver": "delivery_date - order_date in days.",
 "purchase_orders_header.is_cancelled": "TRUE when the PO status is Cancelled.",
}

# Table grain / purpose for raw tables (from the plan, section 4.1)
RAW_GRAIN = {
 "branches": "One row per branch / warehouse (6).",
 "products": "One row per heavy-machinery spare part (30).",
 "inventory_master": "One row per product x branch (180): stock parameters and bin.",
 "suppliers": "One row per China-based supplier (8).",
 "customers": "One row per customer (500).",
 "sales_orders_header": "One row per sales order (20,000).",
 "sales_orders_lines": "One row per sales order line (130,402).",
 "invoices": "One row per invoice for delivered orders (18,033); identified by so_id.",
 "payments": "One row per payment receipt (19,257).",
 "purchase_orders_header": "One row per purchase order (24,000).",
 "purchase_orders_lines": "One row per purchase order line (155,495).",
 "stock_ledger": "One row per stock movement (237,230): IN, OUT or ADJUSTMENT.",
}

# Known data-quality notes per raw column (from the Week 2 issue log)
QUALITY = {
 "invoices.invoice_id": "NOT unique: 197 surplus rows (IDs re-issued). Join on so_id.",
 "invoices.payment_status": "Unreliable: 67 invoices stored Paid are only partly paid. Use payment_status_derived.",
 "payments.payment_id": "NOT unique: 202 surplus rows (IDs re-issued).",
 "customers.total_purchase_value": "Unreliable: about 4% of recomputed sales, correlation -0.08. Use the recomputed value.",
 "customers.region": "22 customers sit in Central / North East, where no branch exists.",
 "customers.pincode": "Stored as a number in raw; text (6 characters) in the clean layer.",
 "suppliers.pincode": "Stored as a number in raw; text (6 characters) in the clean layer.",
 "suppliers.china_tax_id": "City prefix does not match the supplier city (no analytical impact).",
 "branches.warehouse_capacity": "Text with a unit; use warehouse_capacity_sqft.",
 "products.dimensions_cm": "Text 'LxWxH'; parsed into length, width, height and volume.",
 "products.reorder_level": "Company-level; branch-level values in inventory_master are used for stock health.",
 "products.safety_stock": "Company-level; branch-level values in inventory_master are used for stock health.",
 "products.max_stock_level": "Company-level; branch-level values in inventory_master are used for stock health.",
 "inventory_master.current_stock": "Above max_stock in all 180 pairs; differs from the ledger balance in 14 pairs.",
 "purchase_orders_header.received_date": "Empty for the 2,370 cancelled POs (valid, not an error).",
 "sales_orders_header.delivery_date": "Placeholder date on the 1,967 cancelled orders; do not use for delivery measures.",
 "stock_ledger.quantity": "Always positive; ADJUSTMENT direction (gain or loss) is not stored.",
 "stock_ledger.running_balance": "Order of movements within a day is not stored, so the balance cannot be re-sorted.",
 "stock_ledger.movement_id": "Sorts as text, not by time.",
 "sales_orders_lines.unit_price": "Stored as a whole number; line arithmetic reconciles exactly.",
 "sales_orders_lines.line_total": "Stored as a whole number; line arithmetic reconciles exactly.",
}

STAR_SOURCE = {"dim_product": "products", "dim_branch": "branches",
               "dim_customer": "customers", "dim_supplier": "suppliers"}
VERIFIED_TABLES = {"sales_orders_header", "sales_orders_lines", "invoices", "payments",
                   "purchase_orders_header", "purchase_orders_lines", "stock_ledger"}

**Reading the output.** No output: the cell only defines the dictionaries.

## 2. Read the structure from the database

**Objective.** List every column of every table in the four layers, with its type and any description stored in the database.

* **Raw** columns come from schema `main`.
* **Clean** columns are those present in the Week 2 Parquet file but not in the raw table.
* **Star** and **Feature** columns come from schemas `star` and `feat`, including the descriptions written by `COMMENT ON`.

In [3]:
STAR_ORDER = ["dim_date", "dim_product", "dim_branch", "dim_customer", "dim_supplier",
              "fact_sales_line", "fact_purchase_line", "fact_stock_movement", "fact_invoice_payment"]
FEAT_ORDER = ["feat_purchase_order", "feat_supplier", "feat_sales_order", "feat_invoice",
              "feat_customer", "feat_product", "feat_monthly_demand", "feat_demand_variability"]


def db_columns(schema):
    return con.execute("""
        SELECT table_name, column_name, data_type, comment, column_index
        FROM duckdb_columns() WHERE schema_name = ?
        ORDER BY table_name, column_index""", [schema]).df()


records = []


def add(layer, table, column, dtype, comment, order, expr):
    records.append(dict(layer=layer, table=table, column=column, data_type=dtype,
                        db_comment=comment if isinstance(comment, str) else None,
                        col_order=order, from_expr=expr))


raw_cols = db_columns("main")
for t in RAW_GRAIN:
    g = raw_cols[raw_cols["table_name"] == t]
    for r in g.itertuples():
        add("Raw", t, r.column_name, r.data_type, None, r.column_index, f"main.{t}")
    pq = f"data/processed/{t}.parquet"
    if os.path.exists(pq):
        have = set(g["column_name"])
        d = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{pq}')").df()
        for i, r in enumerate(d.itertuples()):
            if r.column_name not in have:
                add("Clean", t, r.column_name, r.column_type, None, 1000 + i,
                    f"read_parquet('{pq}')")

for schema, layer, order in (("star", "Star", STAR_ORDER), ("feat", "Feature", FEAT_ORDER)):
    cc = db_columns(schema)
    for t in order:
        for r in cc[cc["table_name"] == t].itertuples():
            add(layer, t, r.column_name, r.data_type, r.comment, r.column_index, f"{schema}.{t}")

cols_df = pd.DataFrame(records)
cols_df.groupby("layer", sort=False).agg(tables=("table", "nunique"), columns=("column", "count"))

,tables,columns
layer,,
Raw,12,134
Clean,8,18
Star,9,150
Feature,8,138


**Reading the output.** The number of tables and columns found in each layer. Expect **Raw: 12 tables, 134 columns**; **Clean:** the columns added in Week 2 (about 18); **Star:** 9 tables; **Feature:** 8 tables.

## 3. Measure values, ranges and null counts

**Objective.** For every column, record how many values are empty, how many are distinct, and what values it holds.

| Column kind | What is recorded |
|---|---|
| Label with 12 or fewer distinct values (text or TRUE/FALSE) | The **allowed values**, e.g. `Delivered, Cancelled` |
| Text with more than 12 distinct values (IDs, names) | The number of distinct values |
| Number or date | The **range**, minimum to maximum |

One query per table measures all its columns in a single pass over the data.

In [4]:
def fmt(x):
    """Format a minimum/maximum value for display."""
    if x is None:
        return None
    try:
        v = float(x)
        return f"{v:,.0f}" if v == int(v) else f"{v:,.4f}".rstrip("0").rstrip(".")
    except (ValueError, OverflowError):
        return str(x)[:19]


def label_values(expr, col):
    rows = con.execute(f'SELECT DISTINCT CAST("{col}" AS VARCHAR) FROM {expr} '
                       f'WHERE "{col}" IS NOT NULL ORDER BY 1').fetchall()
    return ", ".join(r[0] for r in rows)


def stats_for(expr, cols):
    """Return {column: (n_rows, non_null, distinct, min, max)} for a list of columns."""
    out = {}
    try:
        sel = ["COUNT(*)"]
        for c in cols:
            q = f'"{c}"'
            sel += [f"COUNT({q})", f"COUNT(DISTINCT {q})",
                    f"CAST(MIN({q}) AS VARCHAR)", f"CAST(MAX({q}) AS VARCHAR)"]
        row = con.execute(f"SELECT {', '.join(sel)} FROM {expr}").fetchone()
        for i, c in enumerate(cols):
            o = 1 + 4 * i
            out[c] = (row[0], row[o], row[o + 1], row[o + 2], row[o + 3])
    except Exception:                       # fall back to one query per column
        for c in cols:
            q = f'"{c}"'
            try:
                r = con.execute(f"SELECT COUNT(*), COUNT({q}), COUNT(DISTINCT {q}), "
                                f"CAST(MIN({q}) AS VARCHAR), CAST(MAX({q}) AS VARCHAR) FROM {expr}").fetchone()
            except Exception:
                r = (None, None, None, None, None)
            out[c] = r
    return out


profiles = []
for (layer, table), g in cols_df.groupby(["layer", "table"], sort=False):
    expr = g["from_expr"].iloc[0]
    st = stats_for(expr, list(g["column"]))
    for r in g.itertuples():
        n, nn, nd, mn, mx = st[r.column]
        if n is None:
            profiles.append((layer, table, r.column, None, None, "not measured"))
            continue
        if r.data_type in ("VARCHAR", "BOOLEAN"):
            vals = label_values(expr, r.column) if nd is not None and nd <= 12 else f"{nd:,} distinct values"
        else:
            vals = f"{fmt(mn)} to {fmt(mx)}" if mn is not None else ""
        profiles.append((layer, table, r.column, n - nn, nd, vals))

prof_df = pd.DataFrame(profiles, columns=["layer", "table", "column",
                                          "null_count", "distinct_count", "values_or_range"])
prof_df.sample(8, random_state=1)

,layer,table,column,null_count,distinct_count,values_or_range
283,Star,fact_stock_movement,reference_id,0,43709,"43,709 distinct values"
369,Feature,feat_invoice,is_paid_late,5888,2,"false, true"
207,Star,dim_customer,customer_id,0,500,500 distinct values
300,Star,fact_invoice_payment,first_payment_date_key,2191,2237,"20,190,109 to 20,250,320"
429,Feature,feat_demand_variability,months_observed,0,1,72 to 72
422,Feature,feat_monthly_demand,order_lines,0,29,0 to 30
47,Raw,inventory_master,current_stock,0,180,"88,853 to 121,013"
350,Feature,feat_sales_order,order_year,0,6,"2,019 to 2,024"


**Reading the output.** Eight random columns with their measured null count, distinct count and values or range. For example a label column shows its allowed values (`Delivered, Cancelled`), a date column its first and last date, and an ID column its count of distinct values. Any `not measured` entry means the query for that column failed; none are expected.

## 4. Combine structure, measurements and definitions

**Objective.** Build the final dictionary table, one row per column.

**Where each definition comes from** (first match wins):

1. The description stored in the database (star and feature layers).
2. `CLEAN_DEFS` for columns added by the Week 2 cleaning.
3. `RAW_DEFS` for raw columns. Dimension columns copied from a raw table inherit its definition (`STAR_SOURCE`).

**Status** says how far to trust a definition: **Verified** (checked in Weeks 1-2), **Inferred** (from the column name; confirm with the official guide) or **Defined in** the Week 2 cleaning or Week 3 build.

In [5]:
ID_COLS = {"branch_id", "product_id", "customer_id", "supplier_id", "so_id", "po_id",
           "invoice_id", "payment_id", "movement_id", "line_number"}


def resolve(row):
    layer, table, col = row["layer"], row["table"], row["column"]
    src = STAR_SOURCE.get(table, table) if layer == "Star" else table
    key = f"{src}.{col}"
    if isinstance(row["db_comment"], str) and row["db_comment"]:
        return row["db_comment"], "Defined in Week 3 build", False, ""
    if key in CLEAN_DEFS:
        return CLEAN_DEFS[key], "Defined in Week 2 cleaning", layer == "Star", QUALITY.get(key, "")
    if key in RAW_DEFS:
        verified = src in VERIFIED_TABLES or col in ID_COLS
        status = "Verified (Weeks 1-2 checks)" if verified else "Inferred - confirm with Fields Documentation"
        return RAW_DEFS[key], status, layer == "Star", QUALITY.get(key, "")
    return "", "MISSING", False, ""


res = cols_df.apply(resolve, axis=1, result_type="expand")
res.columns = ["definition", "definition_status", "inherited", "quality_notes"]

dict_df = pd.concat([cols_df, res], axis=1).merge(
    prof_df, on=["layer", "table", "column"], how="left")
dict_df["source_table"] = [STAR_SOURCE.get(t, t) if l == "Star" else t
                           for l, t in zip(dict_df["layer"], dict_df["table"])]

DICT_COLUMNS = ["layer", "table", "column", "data_type", "definition", "definition_status",
                "values_or_range", "null_count", "distinct_count", "quality_notes", "source_table"]

missing = dict_df[dict_df["definition"] == ""]
print(f"{len(dict_df)} columns in the dictionary; {len(missing)} without a definition.")
display(dict_df.groupby(["layer", "definition_status"]).size().rename("columns").to_frame())
missing[["layer", "table", "column"]]

440 columns in the dictionary; 0 without a definition.


columns
layer   definition_status                                    
Clean   Defined in Week 2 cleaning                         18
Feature Defined in Week 3 build                           138
Raw     Inferred - confirm with Fields Documentation       64
        Verified (Weeks 1-2 checks)                        70
Star    Defined in Week 2 cleaning                          6
        Defined in Week 3 build                            82
        Inferred - confirm with Fields Documentation       57
        Verified (Weeks 1-2 checks)                         5

,layer,table,column


**Reading the output.** The first line gives the total number of columns and how many lack a definition. **That number should be 0.** The table groups columns by layer and status: most raw columns are **Inferred** or **Verified**, star and feature columns are **Defined in Week 3 build**. The table at the bottom lists any column with no definition; it should be **empty**. If a column appears, add it to `RAW_DEFS` or `CLEAN_DEFS` in section 1 and re-run.

## 5. Tables overview

**Objective.** One row per table: layer, what a row represents (its grain), row count and column count. It is the map of the whole project.

In [6]:
table_rows = []
for t in RAW_GRAIN:
    n_rows = con.execute(f"SELECT COUNT(*) FROM main.{t}").fetchone()[0]
    g = dict_df[dict_df["table"] == t]
    added = int((g["layer"] == "Clean").sum())
    desc = RAW_GRAIN[t] + (f" Clean layer adds {added} column(s)." if added else "")
    table_rows.append(("Raw", t, desc, n_rows, int((g["layer"] == "Raw").sum())))

for schema, layer, order in (("star", "Star", STAR_ORDER), ("feat", "Feature", FEAT_ORDER)):
    comments = dict(con.execute(
        "SELECT table_name, comment FROM duckdb_tables() WHERE schema_name = ?", [schema]).fetchall())
    for t in order:
        n_rows = con.execute(f"SELECT COUNT(*) FROM {schema}.{t}").fetchone()[0]
        n_cols = int((dict_df["table"] == t).sum())
        table_rows.append((layer, t, comments.get(t) or "", n_rows, n_cols))

tables_df = pd.DataFrame(table_rows, columns=["layer", "table", "description", "rows", "columns"])
tables_df

,layer,table,description,rows,columns
0,Raw,branches,One row per branch / warehouse (6). Clean layer adds 1 column(s).,6,13
1,Raw,products,One row per heavy-machinery spare part (30). Clean layer adds 4 co...,30,23
2,Raw,inventory_master,One row per product x branch (180): stock parameters and bin. Clea...,180,8
3,Raw,suppliers,One row per China-based supplier (8).,8,12
4,Raw,customers,One row per customer (500). Clean layer adds 3 column(s).,500,15
5,Raw,sales_orders_header,"One row per sales order (20,000). Clean layer adds 2 column(s).",20000,11
6,Raw,sales_orders_lines,"One row per sales order line (130,402).",130402,9
7,Raw,invoices,"One row per invoice for delivered orders (18,033); identified by s...",18033,10
8,Raw,payments,"One row per payment receipt (19,257). Clean layer adds 1 column(s).",19257,5
9,Raw,purchase_orders_header,"One row per purchase order (24,000). Clean layer adds 1 column(s).",24000,10


**Reading the output.** 29 rows: 12 raw tables, 9 star tables, 8 feature tables. Each has a one-line description of what one row represents, its row count and its column count. Check the counts against your earlier notebooks (for example `fact_sales_line` 130,402 and `feat_monthly_demand` 12,960).

## 6. Write the dictionary files

**Objective.** Save the dictionary as an **Excel workbook** (all details, filterable) and a **Word document** (formatted for reading and review).

**Writers.** The cell below saves the writing code as `src/dictionary_writers.py`, the same pattern as `src/features.py`.

| File | Content |
|---|---|
| `docs/data_dictionary.xlsx` | Sheets: About, Tables, Data_Dictionary (every column, with status, values, nulls, notes), Issue_Log |
| `docs/data_dictionary.docx` | Purpose, how to read, tables overview, then one section per table; star dimensions list only their own columns; appendix with the Week 2 decisions |

**Word format** follows the team standard: Times New Roman 12 pt, black text, justified paragraphs, 1.5 line spacing, on A4 landscape. Table cells use 10 pt, left aligned, so the wide field tables stay readable; change `table_pt` to 12 if the format must be strict.

**Requirements.** `pip install openpyxl python-docx` (once).

In [9]:
from docx import Document
doc = Document()

In [10]:
%%writefile src/dictionary_writers.py
import os
import pandas as pd

# ------------------------------- Excel --------------------------------------
def write_excel(path, about_rows, tables_df, dict_df, issues_df=None):
    """Write the dictionary workbook: About, Tables, Data_Dictionary (+ Issue_Log)."""
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        pd.DataFrame(about_rows, columns=["Item", "Detail"]).to_excel(xw, sheet_name="About", index=False)
        tables_df.to_excel(xw, sheet_name="Tables", index=False)
        dict_df.to_excel(xw, sheet_name="Data_Dictionary", index=False)
        if issues_df is not None:
            issues_df.to_excel(xw, sheet_name="Issue_Log", index=False)
        head_fill = PatternFill("solid", fgColor="D9D9D9")
        for ws in xw.book.worksheets:
            for cell in ws[1]:
                cell.font = Font(bold=True, name="Times New Roman", size=11)
                cell.fill = head_fill
                cell.alignment = Alignment(vertical="center", wrap_text=True)
            for i, col in enumerate(ws.columns, start=1):
                longest = max((len(str(c.value)) for c in col if c.value is not None), default=10)
                ws.column_dimensions[get_column_letter(i)].width = min(max(longest + 2, 10), 60)
                for c in col[1:]:
                    c.alignment = Alignment(vertical="top", wrap_text=True)
                    c.font = Font(name="Times New Roman", size=11)
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
    return path


# -------------------------------- Word --------------------------------------
def write_word(path, info_rows, tables_df, dict_df, issues_df=None, table_pt=10):
    """Write the data dictionary as a Word document in the team format:
    Times New Roman 12 pt, black text, justified paragraphs, 1.5 line spacing.
    (Table cells use `table_pt`, left aligned, so wide tables stay readable.)"""
    from docx import Document
    from docx.enum.section import WD_ORIENT
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn
    from docx.shared import Cm, Pt, RGBColor

    FONT = "Times New Roman"
    doc = Document()

    # ---- page: A4 landscape -------------------------------------------------
    sec = doc.sections[0]
    sec.orientation = WD_ORIENT.LANDSCAPE
    sec.page_width, sec.page_height = Cm(29.7), Cm(21.0)
    sec.left_margin = sec.right_margin = Cm(2.0)
    sec.top_margin = sec.bottom_margin = Cm(2.0)

    # ---- styles ---------------------------------------------------------------
    def style_font(style, size, bold=False):
        style.font.name = FONT
        style.font.size = Pt(size)
        style.font.bold = bold
        style.font.italic = False
        style.font.color.rgb = RGBColor(0, 0, 0)
        rpr = style.element.get_or_add_rPr()
        rfonts = rpr.find(qn("w:rFonts"))
        if rfonts is None:
            rfonts = OxmlElement("w:rFonts")
            rpr.append(rfonts)
        for attr in ("w:asciiTheme", "w:hAnsiTheme", "w:eastAsiaTheme", "w:cstheme"):
            rfonts.attrib.pop(qn(attr), None)
        for attr in ("w:ascii", "w:hAnsi", "w:eastAsia", "w:cs"):
            rfonts.set(qn(attr), FONT)

    normal = doc.styles["Normal"]
    style_font(normal, 12)
    normal.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    normal.paragraph_format.line_spacing = 1.5
    normal.paragraph_format.space_after = Pt(6)
    for name, size in (("Title", 22), ("Heading 1", 16), ("Heading 2", 13)):
        st = doc.styles[name]
        style_font(st, size, bold=True)
        st.paragraph_format.alignment = (WD_ALIGN_PARAGRAPH.CENTER if name == "Title"
                                         else WD_ALIGN_PARAGRAPH.LEFT)
        st.paragraph_format.keep_with_next = True
        st.paragraph_format.space_before = Pt(12)
        st.paragraph_format.space_after = Pt(6)
        st.paragraph_format.line_spacing = 1.15
        # remove the blue bottom border that the Title style carries
        ppr = st.element.get_or_add_pPr()
        for b in ppr.findall(qn("w:pBdr")):
            ppr.remove(b)

    # ---- helpers --------------------------------------------------------------
    def shade(cell, fill="D9D9D9"):
        tcpr = cell._tc.get_or_add_tcPr()
        shd = OxmlElement("w:shd")
        shd.set(qn("w:val"), "clear")
        shd.set(qn("w:color"), "auto")
        shd.set(qn("w:fill"), fill)
        tcpr.append(shd)

    def cell_text(cell, text, bold=False):
        p = cell.paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.LEFT
        p.paragraph_format.line_spacing = 1.0
        p.paragraph_format.space_after = Pt(0)
        run = p.add_run("" if text is None or (isinstance(text, float) and pd.isna(text)) else str(text))
        run.font.size = Pt(table_pt)
        run.bold = bold

    def add_table(df, widths_cm):
        t = doc.add_table(rows=1, cols=len(df.columns))
        t.style = "Table Grid"
        t.autofit = False
        for i, name in enumerate(df.columns):
            cell_text(t.rows[0].cells[i], name, bold=True)
            shade(t.rows[0].cells[i])
        trpr = t.rows[0]._tr.get_or_add_trPr()          # repeat header on each page
        hdr = OxmlElement("w:tblHeader")
        hdr.set(qn("w:val"), "true")
        trpr.append(hdr)
        for _, r in df.iterrows():
            cells = t.add_row().cells
            for i, v in enumerate(r):
                cell_text(cells[i], v)
        for row in t.rows:
            for i, w in enumerate(widths_cm):
                row.cells[i].width = Cm(w)
        for i, w in enumerate(widths_cm):
            t.columns[i].width = Cm(w)
        doc.add_paragraph()
        return t

    def para(text):
        return doc.add_paragraph(text)

    # ---- footer with page number ------------------------------------------------
    fp = sec.footer.paragraphs[0]
    fp.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = fp.add_run("Heavy Supplier, Inventory & Warehouse Analytics - Data Dictionary - Page ")
    run.font.size = Pt(10)
    r2 = fp.add_run()
    r2.font.size = Pt(10)
    for kind, txt in (("begin", None), (None, "PAGE"), ("end", None)):
        if kind:
            el = OxmlElement("w:fldChar")
            el.set(qn("w:fldCharType"), kind)
        else:
            el = OxmlElement("w:instrText")
            el.set(qn("xml:space"), "preserve")
            el.text = txt
        r2._r.append(el)

    # ---- content ----------------------------------------------------------------
    doc.add_paragraph("Data Dictionary", style="Title")
    doc.add_paragraph("Heavy Supplier, Inventory & Warehouse Analytics", style="Title")
    add_table(pd.DataFrame(info_rows, columns=["Item", "Detail"]), [6.0, 19.7])

    doc.add_heading("1. Purpose", level=1)
    para("This document defines every field used in the project: the 12 raw tables, the columns "
         "added by the Week 2 cleaning layer, the star schema (5 dimensions and 4 facts) and the "
         "feature tables built in Week 3. It lets any team member or reviewer see what a field "
         "means, where it comes from, what values it holds and which data-quality issues affect it.")

    doc.add_heading("2. How to read this dictionary", level=1)
    para("Each field row shows the column name, its data type, a definition, the allowed values "
         "(for labels) or the observed range (for numbers and dates), the number of empty values "
         "and any known quality note. Values, ranges and null counts were measured from the "
         "database when this document was generated.")
    para("The status of a definition is one of: Verified (the field was checked in the Week 1-2 "
         "profiling and reconciliation); Inferred (the meaning is taken from the column name and "
         "must be confirmed against the official Fields Documentation, which was not available); "
         "Defined in the Week 2 cleaning or the Week 3 build (the field was created by this project). "
         "The status of every field is given in the Excel version of this dictionary.")

    doc.add_heading("3. Layers and tables", level=1)
    add_table(tables_df[["layer", "table", "description", "rows", "columns"]]
              .rename(columns={"layer": "Layer", "table": "Table", "description": "Description",
                               "rows": "Rows", "columns": "Columns"}),
              [2.6, 5.2, 14.0, 2.0, 1.9])

    def field_table(df):
        out = pd.DataFrame({
            "Column": df["column"], "Type": df["data_type"], "Definition": df["definition"],
            "Values / range": df["values_or_range"], "Nulls": df["null_count"],
            "Notes": df["quality_notes"]})
        add_table(out, [4.0, 2.2, 8.6, 4.8, 1.4, 4.7])

    def section_for(title, layer_names, only_defined=False):
        part = dict_df[dict_df["layer"].isin(layer_names)]
        for tname, g in part.groupby("table", sort=False):
            doc.add_heading(tname, level=2)
            desc = tables_df.loc[tables_df["table"] == tname, "description"]
            if len(desc):
                para(str(desc.iloc[0]))
            if only_defined and bool(g["inherited"].any()):
                g = g[~g["inherited"].astype(bool)]
                para("Columns copied unchanged from the source table are not repeated here; "
                     "see the source table in section 4.")
            field_table(g)

    doc.add_heading("4. Raw and cleaned tables", level=1)
    para("Columns marked [Clean layer] in the Notes column were added or changed by the Week 2 "
         "cleaning pipeline. All other columns are as received.")
    raw = dict_df[dict_df["layer"].isin(["Raw", "Clean"])].copy()
    raw["quality_notes"] = [("[Clean layer] " + (n or "")).strip() if l == "Clean" else n
                            for l, n in zip(raw["layer"], raw["quality_notes"])]
    for tname, g in raw.groupby("table", sort=False):
        doc.add_heading(tname, level=2)
        desc = tables_df.loc[(tables_df["table"] == tname) & (tables_df["layer"] == "Raw"), "description"]
        if len(desc):
            para(str(desc.iloc[0]))
        field_table(g)

    doc.add_heading("5. Star schema (schema star)", level=1)
    para("Facts hold the measurable events; dimensions describe who, what, where and when. "
         "Natural keys are used. Dates are stored as integer keys (YYYYMMDD) joining to dim_date.")
    section_for("star", ["Star"], only_defined=True)

    doc.add_heading("6. Feature tables (schema feat)", level=1)
    para("Derived measures built in Week 3 for KPIs, segmentation, forecasting and optimisation.")
    section_for("feat", ["Feature"])

    if issues_df is not None and len(issues_df):
        doc.add_heading("Appendix A. Data-quality decisions (Week 2 issue log)", level=1)
        para("Each issue found in Week 2 has one decision: FIX (corrected in the clean layer), "
             "FLAG (kept and marked, needs a business answer), ACCEPT (valid or explainable) or "
             "EXCLUDE (kept but left out of specific analyses through a flag column).")
        keep = [c for c in ["Issue_ID", "Table_or_Column", "Severity", "Issue", "Decision"]
                if c in issues_df.columns]
        add_table(issues_df[keep], [1.8, 6.5, 2.0, 11.4, 2.5][:len(keep)])

    doc.save(path)
    return path

Overwriting src/dictionary_writers.py


**Reading the output.** `Writing src/dictionary_writers.py`.

In [11]:
sys.path.insert(0, os.getcwd())
import src.dictionary_writers as dw
importlib.reload(dw)

issues_df = None
if os.path.exists("docs/dq_issue_log_v2.csv"):
    issues_df = pd.read_csv("docs/dq_issue_log_v2.csv")

about_rows = [
    ("Project", "Heavy Supplier, Inventory & Warehouse Analytics (CadetX Virtual Work Experience)"),
    ("Prepared by", "Keren Oyem, Data Analyst / Data Scientist"),
    ("Task", "A6 Data Dictionary & Documentation (Week 3)"),
    ("Generated on", datetime.date.today().isoformat()),
    ("Source", "warehouse.duckdb (schemas main, star, feat) and data/processed/*.parquet"),
    ("Layers", "Raw, Clean (Week 2 additions), Star (Week 3), Feature (Week 3)"),
    ("Status values", "Verified = checked in Weeks 1-2; Inferred = from column name, confirm with Fields Documentation; "
                      "Defined in = created by this project"),
    ("Limitation", "Fields Documentation.docx was not available; raw master-table definitions are inferred."),
]

xlsx_path = dw.write_excel("docs/data_dictionary.xlsx", about_rows, tables_df,
                           dict_df[DICT_COLUMNS], issues_df)
docx_path = dw.write_word("docs/data_dictionary.docx", about_rows, tables_df, dict_df, issues_df)
print("Saved:", xlsx_path, "and", docx_path)

Saved: docs/data_dictionary.xlsx and docs/data_dictionary.docx


**Reading the output.** `Saved: docs/data_dictionary.xlsx and docs/data_dictionary.docx`. Open both: the Excel file has a filter on every column (filter `definition_status` for *Inferred* to see what still needs confirming); the Word file starts with the purpose and tables overview.

## 7. Check the result

**Objective.** Confirm the dictionary is complete and both files are valid. All checks should be `PASSED`.

In [12]:
from docx import Document

doc = Document(docx_path)
xl = pd.ExcelFile(xlsx_path)
xl_dict = xl.parse("Data_Dictionary")

checks = [
    ("Every column has a definition", 0, len(missing)),
    ("Raw columns documented (134 expected)", 134, int((dict_df["layer"] == "Raw").sum())),
    ("Star tables documented", 9, int(dict_df.loc[dict_df["layer"] == "Star", "table"].nunique())),
    ("Feature tables documented", 8, int(dict_df.loc[dict_df["layer"] == "Feature", "table"].nunique())),
    ("No column left unmeasured", 0, int((dict_df["values_or_range"] == "not measured").sum())),
    ("Excel rows = dictionary rows", len(dict_df), len(xl_dict)),
    ("Excel sheets (About, Tables, Data_Dictionary, Issue_Log)", 4 if issues_df is not None else 3, len(xl.sheet_names)),
    ("Word document has tables", 1, int(len(doc.tables) > 0)),
]
check_df = pd.DataFrame(checks, columns=["check", "expected", "actual"])
check_df["status"] = ["PASSED" if e == a else "FAILED" for e, a in zip(check_df["expected"], check_df["actual"])]
display(check_df)
print(f"{(check_df.status == 'PASSED').sum()} of {len(check_df)} checks passed.")
for p in (xlsx_path, docx_path):
    print(f"{p}: {os.path.getsize(p) / 1024:,.0f} KB")
assert (check_df.status == "PASSED").all(), "A dictionary check failed - see the table above."

,check,expected,actual,status
0,Every column has a definition,0,0,PASSED
1,Raw columns documented (134 expected),134,134,PASSED
2,Star tables documented,9,9,PASSED
3,Feature tables documented,8,8,PASSED
4,No column left unmeasured,0,0,PASSED
5,Excel rows = dictionary rows,440,440,PASSED
6,"Excel sheets (About, Tables, Data_Dictionary, Issue_Log)",4,4,PASSED
7,Word document has tables,1,1,PASSED


8 of 8 checks passed.
docs/data_dictionary.xlsx: 46 KB
docs/data_dictionary.docx: 62 KB


**Reading the output.** One row per check with expected and actual values; all should be `PASSED`. A failed definition check means a column is missing from the library; a failed row-count check means the Excel file does not match the dictionary table. The last lines give the size of each file.

In [13]:
# Which definitions still need confirming against the official Fields Documentation?
(dict_df[dict_df["layer"].isin(["Raw", "Clean"])]
    .groupby("definition_status").size().rename("columns").to_frame())

,columns
definition_status,
Defined in Week 2 cleaning,18
Inferred - confirm with Fields Documentation,64
Verified (Weeks 1-2 checks),70


**Reading the output.** How many raw and clean columns are **Verified**, **Inferred** or **Defined in the Week 2 cleaning**. The Inferred ones (mostly in the five master tables) are the to-do list for when `Fields Documentation.docx` arrives.

## 8. Summary

| Item | Result |
|---|---|
| Coverage | Every column of the raw, clean, star and feature layers has a definition |
| Measured | Null count, distinct count and values or range for every column |
| Quality notes | Linked to the Week 2 issue log (duplicate IDs, unreliable stored fields, placeholders) |
| Files | `docs/data_dictionary.xlsx` and `docs/data_dictionary.docx` |
| Open point | Inferred definitions to confirm with the official Fields Documentation |

**Status: COMPLETED** once all checks pass.

**Next.** Compose the Phase 1 summary report (`reports/phase1_summary.docx`) from the KPI tracker, then the sprint notes and submission.

In [14]:
con.close()
print("Connection closed.")

Connection closed.
